# Classification Evaluation Metrics — from scratch

For a binary problem every prediction falls in one cell of the **confusion matrix**:

| | Predicted positive | Predicted negative |
|---|---|---|
| **Actual positive** | TP (true positive) | FN (false negative) |
| **Actual negative** | FP (false positive) | TN (true negative) |

| Metric | Formula | Question it answers |
|---|---|---|
| **Accuracy** | $\frac{TP+TN}{TP+TN+FP+FN}$ | What fraction of all predictions is correct? |
| **Precision** | $\frac{TP}{TP+FP}$ | Of the predicted positives, how many are real? |
| **Recall** (sensitivity, TPR) | $\frac{TP}{TP+FN}$ | Of the real positives, how many did we find? |
| **F1-score** | $2\cdot\frac{P\cdot R}{P+R}$ | Harmonic mean of precision and recall |
| **FPR** | $\frac{FP}{FP+TN}$ | Of the real negatives, how many did we wrongly flag? |

The **ROC curve** plots TPR vs FPR as the decision threshold sweeps from $+\infty$ to $-\infty$.
**AUC** (area under it) equals the probability that a random positive is scored higher than a random negative.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

In [ ]:
def make_blobs(n_samples=300, centers=3, cluster_std=1.0, n_features=2,
               center_box=(-10, 10), random_state=None):
    """Isotropic Gaussian blobs (re-implementation of sklearn's make_blobs)."""
    rng = np.random.RandomState(random_state)
    if isinstance(centers, int):
        centers = rng.uniform(*center_box, size=(centers, n_features))
    centers = np.asarray(centers, dtype=float)
    k = len(centers)
    counts = [n_samples // k + (1 if i < n_samples % k else 0) for i in range(k)]
    stds = np.broadcast_to(np.asarray(cluster_std, dtype=float), (k,))
    X = np.vstack([rng.normal(c, s, size=(m, centers.shape[1]))
                   for c, s, m in zip(centers, stds, counts)])
    y = np.concatenate([np.full(m, i) for i, m in enumerate(counts)])
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def train_test_split(X, y, test_size=0.25, random_state=None):
    """Shuffle and split arrays into train and test subsets."""
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(X))
    n_test = int(round(len(X) * test_size))
    test, train = idx[:n_test], idx[n_test:]
    return X[train], X[test], y[train], y[test]

## Metric implementations

In [ ]:
def confusion_matrix(y_true, y_pred, labels=None):
    """Rows = actual class, columns = predicted class."""
    labels = np.unique(np.r_[y_true, y_pred]) if labels is None else np.asarray(labels)
    index = {label: i for i, label in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[index[t], index[p]] += 1
    return cm


def accuracy(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))


def precision(y_true, y_pred, positive=1):
    tp = np.sum((y_pred == positive) & (y_true == positive))
    fp = np.sum((y_pred == positive) & (y_true != positive))
    return tp / (tp + fp) if tp + fp else 0.0


def recall(y_true, y_pred, positive=1):
    tp = np.sum((y_pred == positive) & (y_true == positive))
    fn = np.sum((y_pred != positive) & (y_true == positive))
    return tp / (tp + fn) if tp + fn else 0.0


def f1_score(y_true, y_pred, positive=1):
    p, r = precision(y_true, y_pred, positive), recall(y_true, y_pred, positive)
    return 2 * p * r / (p + r) if p + r else 0.0


def classification_report(y_true, y_pred):
    labels = np.unique(np.r_[y_true, y_pred])
    rows = []
    print(f"{'class':>12} {'precision':>10} {'recall':>10} {'f1-score':>10} {'support':>10}")
    for c in labels:
        row = (precision(y_true, y_pred, c), recall(y_true, y_pred, c), f1_score(y_true, y_pred, c), np.sum(y_true == c))
        rows.append(row)
        print(f"{str(c):>12} {row[0]:>10.3f} {row[1]:>10.3f} {row[2]:>10.3f} {row[3]:>10d}")
    rows = np.array(rows, dtype=float)
    support = rows[:, 3]
    macro = rows[:, :3].mean(axis=0)
    weighted = (rows[:, :3] * support[:, None]).sum(axis=0) / support.sum()
    print()
    print(f"{'accuracy':>12} {'':>10} {'':>10} {accuracy(y_true, y_pred):>10.3f} {int(support.sum()):>10d}")
    print(f"{'macro avg':>12} {macro[0]:>10.3f} {macro[1]:>10.3f} {macro[2]:>10.3f} {int(support.sum()):>10d}")
    print(f"{'weighted avg':>12} {weighted[0]:>10.3f} {weighted[1]:>10.3f} {weighted[2]:>10.3f} {int(support.sum()):>10d}")


def roc_curve(y_true, scores):
    """Returns FPR, TPR, thresholds. A sample is predicted positive when score >= threshold."""
    order = np.argsort(-scores, kind="mergesort")
    s, y = scores[order], y_true[order]
    last_of_each_score = np.r_[np.where(np.diff(s))[0], len(s) - 1]   # handle tied scores together
    tps = np.cumsum(y)[last_of_each_score]
    fps = (last_of_each_score + 1) - tps
    tpr = np.r_[0, tps / tps[-1]]
    fpr = np.r_[0, fps / fps[-1]]
    return fpr, tpr, np.r_[np.inf, s[last_of_each_score]]


def auc(x, y):
    """Area under a curve with the trapezoidal rule."""
    return np.sum(np.diff(x) * (y[1:] + y[:-1]) / 2)


def auc_by_ranking(y_true, scores):
    """AUC = P(score of random positive > score of random negative); ties count 1/2."""
    pos, neg = scores[y_true == 1], scores[y_true == 0]
    greater = (pos[:, None] > neg[None, :]).sum()
    ties = (pos[:, None] == neg[None, :]).sum()
    return (greater + 0.5 * ties) / (len(pos) * len(neg))

## A model to evaluate
An imbalanced binary dataset (≈ 80 % negatives) and a from-scratch logistic regression.

In [ ]:
class LogisticRegression:
    def __init__(self, lr=0.1, n_iters=2000):
        self.lr, self.n_iters = lr, n_iters

    def fit(self, X, y):
        self.w, self.b = np.zeros(X.shape[1]), 0.0
        for _ in range(self.n_iters):
            p = self.predict_proba(X)
            self.w -= self.lr * X.T @ (p - y) / len(y)
            self.b -= self.lr * np.mean(p - y)
        return self

    def predict_proba(self, X):
        return 1 / (1 + np.exp(-(X @ self.w + self.b)))

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


rng = np.random.RandomState(0)
n_neg, n_pos = 800, 200
X = np.vstack([rng.normal([0, 0], 1.2, (n_neg, 2)), rng.normal([2, 2], 1.2, (n_pos, 2))])
y = np.r_[np.zeros(n_neg, int), np.ones(n_pos, int)]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

model = LogisticRegression().fit(X_train, y_train)
scores = model.predict_proba(X_test)
y_pred = model.predict(X_test)

## Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print(cm)
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")


def plot_confusion_matrix(cm, labels, ax=None, title="Confusion matrix"):
    ax = ax or plt.gca()
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
    plt.colorbar(im, ax=ax)


plot_confusion_matrix(cm, ["negative (0)", "positive (1)"]); plt.show()

## Accuracy, precision, recall, F1

In [ ]:
print(f"Accuracy : {accuracy(y_test, y_pred):.3f}")
print(f"Precision: {precision(y_test, y_pred):.3f}")
print(f"Recall   : {recall(y_test, y_pred):.3f}")
print(f"F1-score : {f1_score(y_test, y_pred):.3f}\n")
classification_report(y_test, y_pred)

### Why accuracy alone can mislead
A "model" that always predicts the majority class gets high accuracy on imbalanced data but finds no positives.

In [ ]:
always_negative = np.zeros_like(y_test)
print(f"Always-negative baseline: accuracy={accuracy(y_test, always_negative):.3f}, "
      f"recall={recall(y_test, always_negative):.3f}, F1={f1_score(y_test, always_negative):.3f}")

### Precision–recall trade-off
Lowering the threshold catches more positives (higher recall) but adds false alarms (lower precision).

In [ ]:
thresholds = np.linspace(0.05, 0.95, 50)
P = [precision(y_test, (scores >= t).astype(int)) for t in thresholds]
R = [recall(y_test, (scores >= t).astype(int)) for t in thresholds]
F = [f1_score(y_test, (scores >= t).astype(int)) for t in thresholds]
best_t = thresholds[int(np.argmax(F))]

plt.plot(thresholds, P, label="precision"); plt.plot(thresholds, R, label="recall"); plt.plot(thresholds, F, label="F1")
plt.axvline(best_t, ls="--", c="gray", label=f"best F1 threshold = {best_t:.2f}")
plt.xlabel("decision threshold"); plt.legend(); plt.title("Metrics vs threshold"); plt.show()

## ROC curve and AUC

In [ ]:
fpr, tpr, thr = roc_curve(y_test, scores)
roc_auc = auc(fpr, tpr)
print(f"AUC (trapezoid)          = {roc_auc:.4f}")
print(f"AUC (ranking definition) = {auc_by_ranking(y_test, scores):.4f}")

# weaker models for comparison
weak = LogisticRegression().fit(X_train[:, :1], y_train)             # uses only feature 0
weak_scores = weak.predict_proba(X_test[:, :1])
random_scores = np.random.RandomState(1).rand(len(y_test))

plt.figure(figsize=(6.5, 6))
for name, s in [("logistic (2 features)", scores), ("logistic (1 feature)", weak_scores), ("random scores", random_scores)]:
    f, t, _ = roc_curve(y_test, s)
    plt.plot(f, t, lw=2, label=f"{name}: AUC={auc(f, t):.3f}")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="chance")

# mark the point for threshold 0.5
i = np.argmin(np.abs(thr - 0.5))
plt.scatter(fpr[i], tpr[i], s=80, c="red", zorder=5, label="threshold ≈ 0.5")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate (recall)")
plt.title("ROC curves"); plt.legend(loc="lower right"); plt.show()

## Multiclass metrics
For $K$ classes, precision/recall/F1 are computed per class (one-vs-rest) and averaged:
* **macro** — simple mean over classes (each class equally important)
* **weighted** — mean weighted by class support

In [ ]:
X3, y3 = make_blobs(n_samples=600, centers=3, cluster_std=3.0, random_state=11)
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.3, random_state=0)

# a nearest-centroid classifier is enough to produce some confusions
centroids = np.array([X3_train[y3_train == c].mean(0) for c in range(3)])
y3_pred = np.argmin(((X3_test[:, None, :] - centroids[None]) ** 2).sum(-1), axis=1)

classification_report(y3_test, y3_pred)
plot_confusion_matrix(confusion_matrix(y3_test, y3_pred), ["class 0", "class 1", "class 2"],
                      title="3-class confusion matrix"); plt.show()

## Summary — which metric when?
* **Accuracy**: balanced classes, equal error costs.
* **Precision**: false positives are expensive (spam filter deleting real mail).
* **Recall**: false negatives are expensive (disease screening, fraud).
* **F1**: single number balancing P and R, good for imbalanced data.
* **ROC / AUC**: threshold-independent measure of how well the model *ranks* positives above negatives.